# Predicting late deliveries
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-fold walk-forward CV on training rows, choose by mean CV ROC AUC, score holdout once).

**Feature decision:** `claim_paid_eur` is compensation paid on a claim *settled for the consignment*, i.e. recorded after the delivery outcome (it is >0 almost only when late). It is not known at dispatch, so it is excluded as leakage. `consignment` (id) and `dispatched_on` (split column) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train = df.drop(index=hold_idx)
test = df.loc[hold_idx]
train = train.assign(_t=pd.to_datetime(train.dispatched_on)).sort_values('_t')
test = test.assign(_t=pd.to_datetime(test.dispatched_on))
print(len(train), len(test), train._t.max(), test._t.min())
print(train.late_delivery.mean(), test.late_delivery.mean())

In [ ]:
TARGET='late_delivery'
cat=['carrier']; num=['distance_km','weight_kg','parcels']
FEATURES=cat+num
print(train.groupby('carrier')[TARGET].agg(['mean','size']))
print(train[num+[TARGET]].corr()[TARGET])

## Candidates (named before any scoring)

In [ ]:
def prep(scale=True):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates = {
 'baseline_prior': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',prep(False)),('m',RandomForestClassifier(300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Walk-forward CV on training rows (preprocessing fit inside each fold)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
tr = train.reset_index()  # 'index' = 0-based row position in data.csv, ordered by time
X, y = tr[FEATURES], tr[TARGET].to_numpy()
folds = list(TimeSeriesSplit(n_splits=5).split(X))
cv_scores = {}
for name, est in candidates.items():
    cv_scores[name] = [float(roc_auc_score(y[v], clone(est).fit(X.iloc[t], y[t]).predict_proba(X.iloc[v])[:,1])) for t,v in folds]
cv = pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1)
cv.sort_values('mean',ascending=False)

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen, np.mean(cv_scores[chosen]))

## Refit on all training rows, score holdout once

In [ ]:
final = clone(candidates[chosen]).fit(X, y)
p = final.predict_proba(test[FEATURES])[:,1]
holdout_auc = float(roc_auc_score(test[TARGET], p))
print('Holdout ROC AUC:', holdout_auc)

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(i) for i in tr['index'].iloc[t]], [int(i) for i in tr['index'].iloc[v]]] for t,v in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_idx],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))